In [1]:
!pip install -q tensorflow scikit-learn seaborn openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 21.12.2 requires cupy-cuda115, which is not installed.
tfx-bsl 1.12.0 requires google-api-python-client<2,>=1.7.11, but you have google-api-python-client 2.83.0 which is incompatible.
tfx-bsl 1.12.0 requires pyarrow<7,>=6, but you have pyarrow 5.0.0 which is incompatible.
tensorflow-transform 1.12.0 requires pyarrow<7,>=6, but you have pyarrow 5.0.0 which is incompatible.
onnx 1.13.1 requires protobuf<4,>=3.20.2, but you have protobuf 3.19.6 which is incompatible.
apache-beam 2.44.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.6 which is incompatible.


In [2]:
import os
import gc
import zipfile
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    SimpleRNN,
    LSTM,
    GRU,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

warnings.filterwarnings("ignore")

In [7]:
CSV_PATH = "/kaggle/input/datasets/thebluestar/my-flow-data/traffic.csv"

df = pd.read_csv(CSV_PATH)

print(df.head())

print(df.shape)

   timestep  location   flow  occupy  speed
0         1         0  133.0  0.0603   65.8
1         1         1  210.0  0.0589   69.6
2         1         2  124.0  0.0358   65.8
3         1         3  145.0  0.0416   69.6
4         1         4  206.0  0.0493   69.4
(3035520, 5)


In [9]:
features = [
    "flow",
    "occupy",
    "speed"
]

target = "flow"

In [10]:
scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(df[features])

print(scaled_data.shape)

(3035520, 3)


In [11]:
SEQ_LEN = 12

X = []
y = []

for i in range(SEQ_LEN, len(scaled_data)):

    X.append(
        scaled_data[i-SEQ_LEN:i]
    )

    y.append(
        scaled_data[i][0]
    )

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3035508, 12, 3)
y shape: (3035508,)


In [12]:
split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(2428406, 12, 3)
(607102, 12, 3)


In [13]:
OUTPUT_DIR = "/kaggle/working/rnn_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(OUTPUT_DIR)

/kaggle/working/rnn_outputs


In [14]:
model_configs = [

    {
        "name": "SimpleRNN",
        "type": "rnn"
    },

    {
        "name": "LSTM",
        "type": "lstm"
    },

    {
        "name": "GRU",
        "type": "gru"
    }

]

In [15]:
def build_model(model_type):

    model = Sequential()

    if model_type == "rnn":

        model.add(

            SimpleRNN(
                64,
                return_sequences=False,
                input_shape=(
                    X_train.shape[1],
                    X_train.shape[2]
                )
            )

        )

    elif model_type == "lstm":

        model.add(

            LSTM(
                64,
                return_sequences=False,
                input_shape=(
                    X_train.shape[1],
                    X_train.shape[2]
                )
            )

        )

    elif model_type == "gru":

        model.add(

            GRU(
                64,
                return_sequences=False,
                input_shape=(
                    X_train.shape[1],
                    X_train.shape[2]
                )
            )

        )

    model.add(Dropout(0.2))

    model.add(Dense(32, activation="relu"))

    model.add(Dense(1))

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
results_summary = []

for config in model_configs:

    model_name = config["name"]
    model_type = config["type"]

    print("\n" + "="*70)
    print(f"TRAINING: {model_name}")
    print("="*70)

    model = build_model(model_type)

    callbacks = [

        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5
        )

    ]

    history = model.fit(

        X_train,
        y_train,

        validation_data=(X_test, y_test),

        epochs=30,

        batch_size=32,

        verbose=1,

        callbacks=callbacks

    )

    y_pred = model.predict(X_test)

    mse = mean_squared_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    results_summary.append({

        "Model": model_name,

        "MSE": mse,

        "RMSE": rmse,

        "MAE": mae,

        "R2": r2

    })

    model.save(

        os.path.join(
            OUTPUT_DIR,
            f"{model_name}.h5"
        )

    )

    # =====================================
    # LOSS CURVE
    # =====================================

    plt.figure(figsize=(10,5))

    plt.plot(
        history.history["loss"],
        label="Train Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.title(f"{model_name} Loss Curve")

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.legend()

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            OUTPUT_DIR,
            f"{model_name}_loss_curve.png"
        ),

        dpi=300

    )

    plt.close()

    # =====================================
    # PREDICTION CHART
    # =====================================

    plt.figure(figsize=(12,6))

    plt.plot(
        y_test[:200],
        label="Actual"
    )

    plt.plot(
        y_pred[:200],
        label="Predicted"
    )

    plt.title(f"{model_name} Prediction")

    plt.xlabel("Time")

    plt.ylabel("Flow")

    plt.legend()

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            OUTPUT_DIR,
            f"{model_name}_prediction.png"
        ),

        dpi=300

    )

    plt.close()

    gc.collect()


TRAINING: SimpleRNN
Epoch 1/30
75888/75888 [==============================] - 742s 10ms/step - loss: 0.0046 - mae: 0.0507 - val_loss: 0.0028 - val_mae: 0.0390 - lr: 0.0010
Epoch 2/30
75888/75888 [==============================] - 756s 10ms/step - loss: 0.0030 - mae: 0.0407 - val_loss: 0.0021 - val_mae: 0.0344 - lr: 0.0010
Epoch 3/30
75888/75888 [==============================] - 758s 10ms/step - loss: 0.0024 - mae: 0.0363 - val_loss: 0.0019 - val_mae: 0.0316 - lr: 0.0010
Epoch 4/30
75888/75888 [==============================] - 753s 10ms/step - loss: 0.0022 - mae: 0.0345 - val_loss: 0.0018 - val_mae: 0.0311 - lr: 0.0010
Epoch 5/30
75888/75888 [==============================] - 756s 10ms/step - loss: 0.0021 - mae: 0.0335 - val_loss: 0.0016 - val_mae: 0.0293 - lr: 0.0010
Epoch 6/30
59713/75888 [======================>.......] - ETA: 2:29 - loss: 0.0020 - mae: 0.0329

In [ ]:
results_df = pd.DataFrame(results_summary)

results_df = results_df.sort_values(
    by="RMSE"
)

print(results_df)

In [ ]:
results_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "rnn_benchmark_results.csv"
    ),

    index=False

)

results_df.to_excel(

    os.path.join(
        OUTPUT_DIR,
        "rnn_benchmark_results.xlsx"
    ),

    index=False

)

print("Saved CSV and Excel")

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    results_df["Model"],
    results_df["RMSE"]
)

y_min = results_df["RMSE"].min() * 0.95
y_max = results_df["RMSE"].max() * 1.05

plt.ylim(y_min, y_max)

for i, v in enumerate(results_df["RMSE"]):

    plt.text(
        i,
        v,
        f"{v:.4f}",
        ha='center'
    )

plt.title("RMSE Comparison")

plt.ylabel("RMSE")

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "rmse_comparison.png"
    ),

    dpi=300

)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    results_df["Model"],
    results_df["MAE"]
)

y_min = results_df["MAE"].min() * 0.95
y_max = results_df["MAE"].max() * 1.05

plt.ylim(y_min, y_max)

for i, v in enumerate(results_df["MAE"]):

    plt.text(
        i,
        v,
        f"{v:.4f}",
        ha='center'
    )

plt.title("MAE Comparison")

plt.ylabel("MAE")

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "mae_comparison.png"
    ),

    dpi=300

)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    results_df["Model"],
    results_df["R2"]
)

plt.ylim(0.5, 1.0)

for i, v in enumerate(results_df["R2"]):

    plt.text(
        i,
        v,
        f"{v:.4f}",
        ha='center'
    )

plt.title("R2 Comparison")

plt.ylabel("R2")

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "r2_comparison.png"
    ),

    dpi=300

)

plt.show()

In [ ]:
with open(

    os.path.join(
        OUTPUT_DIR,
        "README.txt"
    ),

    "w"

) as f:

    f.write(str(results_df))

print("README saved")

In [ ]:
ZIP_PATH = "/kaggle/working/rnn_traffic_outputs.zip"

with zipfile.ZipFile(

    ZIP_PATH,

    "w",

    zipfile.ZIP_DEFLATED

) as zipf:

    for root, dirs, files in os.walk(OUTPUT_DIR):

        for file in files:

            file_path = os.path.join(root, file)

            arcname = os.path.relpath(
                file_path,
                OUTPUT_DIR
            )

            zipf.write(file_path, arcname)

print("\nZIP SAVED:")
print(ZIP_PATH)